In [ ]:
# In[1]:

import json

# Load the task.json file to understand the alert context
task_directory = 'rca100/cases/t079'
task_file_path = f'{task_directory}/task.json'

with open(task_file_path, 'r') as file:
    task_data = json.load(file)

task_data

```
Out[1]:
```
The alert is titled "inventory接口流量下跌告警" (inventory interface traffic drop alert). It was triggered on April 2, 2026, at 23:07:54 (UTC+8). The alert time window spans from 23:06:32 to 23:09:54 (UTC+8). The entity that triggered the alert is an APM operation named `inventory::/api/v1/inventory/{productId}`. The user's request is to analyze the root cause of this issue.

The original code execution output of IPython Kernel is also provided below for reference:

{'task_id': 't079', 'task_version': '1.0', 'alert_event_id': '2df4be269d71174612851daa32a57939', 'alert_title': 'inventory接口流量下跌告警', 'alert_trigger_time': '2026-04-02T23:07:54+0800', 'alert_window': {'start': '2026-04-02T23:06:32.535684+08:00', 'end': '2026-04-02T23:09:54+08:00'}, 'alert_entity': {'entity_id': '162799adab6799326676266a2c96cf3e', 'entity_name': 'inventory::/api/v1/inventory/{productId}', 'entity_type': 'apm.operation', 'entity_domain': 'apm'}, 'prompt_text': '<code vibeops_object type="alert_event">inventory接口流量下跌告警<alert_event event_id="2df4be269d71174612851daa32a57939" trans_id="kp0t1q7d1p07f5kesfppsu5e56" rule_id="b7853e06-54b7-4409-9eea-9bb82fbfb252" rule_name="inventory接口流量下跌告警" /></code> 帮我分析下根因。', 'workspace': 'rca-benchmark', 'region_id': 'cn-hongkong', 'available_modalities': ['metrics', 'logs', 'traces', 'events', 'alerts', 'topology'], 'scoring_note': 'Output contract (prediction_schema.json) and fault taxonomy (taxonomy.json) will be published in a follow-up release.'}```
```

In [ ]:
# In[2]:

import pandas as pd

# Load the metrics.parquet file
metrics_file_path = f'{task_directory}/metrics.parquet'
metrics_df = pd.read_parquet(metrics_file_path)

# Convert 'time' and 'value' columns to numeric types
metrics_df['time'] = pd.to_numeric(metrics_df['time'], errors='coerce')
metrics_df['value'] = pd.to_numeric(metrics_df['value'], errors='coerce')

# Filter for service-level metrics
service_metrics_df = metrics_df[metrics_df['entity_set'] == 'apm.service.legacy']

# Calculate global thresholds (P95) for each KPI across the entire dataset
global_thresholds = service_metrics_df.groupby('metric')['value'].quantile(0.95).reset_index()
global_thresholds.columns = ['metric', 'P95']

service_metrics_df, global_thresholds

```
Out[2]:
```
The `metrics.parquet` file was successfully loaded, and service-level metrics were filtered. Global thresholds (P95) for key performance indicators (KPIs) across all services were calculated as follows:

- `cpu`: 0.248
- `error`: 27.8
- `latency`: 0.249 seconds
- `mem`: 1.533 GB
- `slow`: 2222 occurrences
- `workload`: 34,312.5 requests

These thresholds represent the 95th percentile values for each KPI across the entire dataset.

The original code execution output of IPython Kernel is also provided below for reference:

(                   time domain          entity_set entity_id     entity_name    metric         value metric_set_id service
73499  1775138792000000    apm  apm.service.legacy                accounting       cpu  6.272872e-03          None    None
73500  1775138792000000    apm  apm.service.legacy                accounting   latency  4.766168e-02          None    None
73501  1775138792000000    apm  apm.service.legacy                accounting       mem  1.005008e+08          None    None
73502  1775138792000000    apm  apm.service.legacy                accounting  workload  3.790000e+02          None    None
73503  1775138792000000    apm  apm.service.legacy                        ad       cpu  2.756008e-03          None    None
...                 ...    ...                 ...       ...             ...       ...           ...           ...     ...
83057  1775142692000000    apm  apm.service.legacy            recommendation  workload  3.140000e+02          None    None
83058  1775142692000000    apm  apm.service.legacy                  shipping       cpu  9.456423e-03          None    None
83059  1775142692000000    apm  apm.service.legacy                  shipping   latency  1.170969e-03          None    None
83060  1775142692000000    apm  apm.service.legacy                  shipping       mem  3.188053e+06          None    None
83061  1775142692000000    apm  apm.service.legacy                  shipping  workload  9.665000e+03          None    None

[9563 rows x 9 columns],      metric           P95
0       cpu  2.483643e-01
1     error  2.780000e+01
2   latency  2.487351e-01
3       mem  1.533042e+09
4      slow  2.222000e+03
5  workload  3.431250e+04)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

import pytz
from datetime import datetime

# Define the alert time window in UTC+8
alert_start = datetime.fromisoformat('2026-04-02T23:06:32.535684+08:00').astimezone(pytz.timezone('Asia/Shanghai'))
alert_end = datetime.fromisoformat('2026-04-02T23:09:54+08:00').astimezone(pytz.timezone('Asia/Shanghai'))

# Convert alert time window to microseconds for filtering
alert_start_us = int(alert_start.timestamp() * 1_000_000)
alert_end_us = int(alert_end.timestamp() * 1_000_000)

# Filter service-level metrics within the alert time window
filtered_service_metrics = service_metrics_df[
    (service_metrics_df['time'] >= alert_start_us) & (service_metrics_df['time'] <= alert_end_us)
]

# Identify anomalies by comparing values against global thresholds (P95)
anomalies = filtered_service_metrics.merge(global_thresholds, on='metric')
anomalies['is_anomalous'] = anomalies['value'] > anomalies['P95']

# Focus on consecutive anomalies for each KPI of each service
consecutive_anomalies = anomalies[anomalies['is_anomalous']].groupby(['entity_name', 'metric']).size().reset_index()
consecutive_anomalies.columns = ['service', 'metric', 'consecutive_anomalies']

filtered_service_metrics, anomalies, consecutive_anomalies

```
Out[3]:
```
The service-level metrics were filtered to the alert time window (23:06:32 to 23:09:54 UTC+8), and anomalies were identified by comparing KPI values against their global P95 thresholds. Consecutive anomalies were detected for the following services and KPIs:

- `frontend`: CPU (1 anomaly), error rate (3 anomalies), workload (3 anomalies)
- `frontend-proxy`: error rate (1 anomaly)
- `load-generator`: memory usage (6 anomalies)

These consecutive anomalies indicate potential issues in these services during the alert time window.

The original code execution output of IPython Kernel is also provided below for reference:

(                   time domain          entity_set entity_id     entity_name    metric         value metric_set_id service
82332  1775142422000000    apm  apm.service.legacy                accounting       cpu  6.248332e-03          None    None
82333  1775142422000000    apm  apm.service.legacy                accounting   latency  2.537869e-02          None    None
82334  1775142422000000    apm  apm.service.legacy                accounting       mem  1.004885e+08          None    None
82335  1775142422000000    apm  apm.service.legacy                accounting  workload  1.801000e+03          None    None
82336  1775142422000000    apm  apm.service.legacy                        ad       cpu  1.488216e-03          None    None
...                 ...    ...                 ...       ...             ...       ...           ...           ...     ...
82765  1775142572000000    apm  apm.service.legacy            recommendation  workload  2.680000e+02          None    None
82766  1775142572000000    apm  apm.service.legacy                  shipping       cpu  8.873475e-03          None    None
82767  1775142572000000    apm  apm.service.legacy                  shipping   latency  1.164348e-03          None    None
82768  1775142572000000    apm  apm.service.legacy                  shipping       mem  3.193515e+06          None    None
82769  1775142572000000    apm  apm.service.legacy                  shipping  workload  7.810000e+03          None    None

[438 rows x 9 columns],                  time domain          entity_set entity_id     entity_name  ...         value  metric_set_id service           P95  is_anomalous
0    1775142422000000    apm  apm.service.legacy                accounting  ...  6.248332e-03           None    None  2.483643e-01         False
1    1775142422000000    apm  apm.service.legacy                accounting  ...  2.537869e-02           None    None  2.487351e-01         False
2    1775142422000000    apm  apm.service.legacy                accounting  ...  1.004885e+08           None    None  1.533042e+09         False
3    1775142422000000    apm  apm.service.legacy                accounting  ...  1.801000e+03           None    None  3.431250e+04         False
4    1775142422000000    apm  apm.service.legacy                        ad  ...  1.488216e-03           None    None  2.483643e-01         False
..                ...    ...                 ...       ...             ...  ...           ...            ...     ...           ...           ...
433  1775142572000000    apm  apm.service.legacy            recommendation  ...  2.680000e+02           None    None  3.431250e+04         False
434  1775142572000000    apm  apm.service.legacy                  shipping  ...  8.873475e-03           None    None  2.483643e-01         False
435  1775142572000000    apm  apm.service.legacy                  shipping  ...  1.164348e-03           None    None  2.487351e-01         False
436  1775142572000000    apm  apm.service.legacy                  shipping  ...  3.193515e+06           None    None  1.533042e+09         False
437  1775142572000000    apm  apm.service.legacy                  shipping  ...  7.810000e+03           None    None  3.431250e+04         False

[438 rows x 11 columns],           service    metric  consecutive_anomalies
0        frontend       cpu                      1
1        frontend     error                      3
2        frontend  workload                      3
3  frontend-proxy     error                      1
4  load-generator       mem                      6)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

# Load the traces.parquet file
traces_file_path = f'{task_directory}/traces.parquet'
traces_df = pd.read_parquet(traces_file_path)

# Convert 'startTime' and 'endTime' to numeric types
traces_df['startTime'] = pd.to_numeric(traces_df['startTime'], errors='coerce')
traces_df['endTime'] = pd.to_numeric(traces_df['endTime'], errors='coerce')

# Convert alert time window to nanoseconds for filtering
alert_start_ns = alert_start.timestamp() * 1_000_000_000
alert_end_ns = alert_end.timestamp() * 1_000_000_000

# Filter traces within the alert time window
filtered_traces = traces_df[
    (traces_df['startTime'] >= alert_start_ns) & (traces_df['endTime'] <= alert_end_ns)
]

# Identify traces involving the 'inventory::/api/v1/inventory/{productId}' operation
inventory_traces = filtered_traces[
    filtered_traces['spanName'] == 'inventory::/api/v1/inventory/{productId}'
]

# Analyze call chains to determine the most downstream faulty service
# A downstream service will have its 'kind' as 'SERVER' and may have error status
downstream_faulty_services = inventory_traces[
    (inventory_traces['kind'] == 'SERVER') & (inventory_traces['statusCode'] != 0)
]

# Group by serviceName to identify the most downstream faulty service
faulty_service_analysis = downstream_faulty_services.groupby('serviceName').size().reset_index(name='error_count')

filtered_traces, inventory_traces, downstream_faulty_services, faulty_service_analysis

```
Out[4]:
```
The `traces.parquet` file was successfully loaded, and traces were filtered to the alert time window (23:06:32 to 23:09:54 UTC+8). However, no traces involving the `inventory::/api/v1/inventory/{productId}` operation were found, and no downstream faulty services were identified in the traces. This suggests that the issue may not have propagated through this specific operation during the alert time window.

The original code execution output of IPython Kernel is also provided below for reference:

(                                 traceId            spanId      parentSpanId kind                                   spanName  ...  __topic__      __tag__:__pack_id__                                             events links    ip
270001  6b3f3bbce424dcfa693ecf490b42b96e  6af1a38ca79713ab  ec55569b210928ce    1                            calculate-quote  ...             8AE7E95982B49865-360DB8  [{"attributes":{},"name":"Calculating quote","...  None  None
270004  6b3f3bbce424dcfa693ecf490b42b96e  ec55569b210928ce  87e9fdc16d01652f    1                                  {closure}  ...             8AE7E95982B49865-360DB8  [{"attributes":{},"name":"Received get quote r...  None  None
270005  6b3f3bbce424dcfa693ecf490b42b96e  87e9fdc16d01652f  9eceab72b96927bd    2                             POST /getquote  ...             8AE7E95982B49865-360DB8                                               None  None  None
270006  3f6a824479624e0ddb3ef949f639fc08  5f8d13a8651b91fa  aaaf2dec0f594dc8    2         POST /oteldemo.CartService/GetCart  ...             8AE7E95982B49865-360DB8  [{"attributes":{},"name":"Fetch cart","timesta...  None  None
270021  b7e1cc64f724264cff03d902b4f13fe9  f6efc6c48ca66f76  7db91bdbbbdb47f4    2  oteldemo.ProductCatalogService/GetProduct  ...             8AE7E95982B49865-360DB8  [{"attributes":{},"name":"Product Found","time...  None  None
...                                  ...               ...               ...  ...                                        ...  ...        ...                      ...                                                ...   ...   ...
339995  b427bc7927c0dfaf69f078ce9f9684bb  339d0b593828a35e  df3d42fb18f860e4    3                     router frontend egress  ...             7939C9365296EE5C-35D6E6                                               None  None  None
339996  cce5b800894dffa655669d91e2fd1c52  7797ffbd3acf0eda  c59129ff96b57e07    3                     router frontend egress  ...             7939C9365296EE5C-35D6E6                                               None  None  None
339997  63412865f4363bc2e5dc1cce59f97bd0  548665f434700b15  ba609539824b70a7    2                           Currency/Convert  ...             7939C9365296EE5C-35D6E6  [{"attributes":{},"name":"Processing currency ...  None  None
339998  90dd4e61f3dd15a95d8d75028b537364  a606e382ea29bd16  13f82480ba71f3c8    2                           Currency/Convert  ...             7939C9365296EE5C-35D6E6  [{"attributes":{},"name":"Processing currency ...  None  None
339999  881908c0e4d9cf04b0f96d5aff1b6bc9  9e3c055cd6d1b758  82dec228bd938d6e    3                     router frontend egress  ...             7939C9365296EE5C-35D6E6                                               None  None  None

[69788 rows x 21 columns], Empty DataFrame
Columns: [traceId, spanId, parentSpanId, kind, spanName, startTime, endTime, duration, serviceName, pid, hostname, statusCode, statusMessage, traceState, resources, attributes, __topic__, __tag__:__pack_id__, events, links, ip]
Index: []

[0 rows x 21 columns], Empty DataFrame
Columns: [traceId, spanId, parentSpanId, kind, spanName, startTime, endTime, duration, serviceName, pid, hostname, statusCode, statusMessage, traceState, resources, attributes, __topic__, __tag__:__pack_id__, events, links, ip]
Index: []

[0 rows x 21 columns], Empty DataFrame
Columns: [serviceName, error_count]
Index: [])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[5]:

# Load the logs.parquet file
logs_file_path = f'{task_directory}/logs.parquet'
logs_df = pd.read_parquet(logs_file_path)

# Convert '_time_' to numeric if necessary
logs_df['_time_'] = pd.to_numeric(logs_df['_time_'], errors='coerce')

# Filter logs within the alert time window
filtered_logs = logs_df[
    (logs_df['_time_'] >= alert_start_us) & (logs_df['_time_'] <= alert_end_us)
]

# Search for error or warning messages related to the specified services
services_of_interest = ['frontend', 'frontend-proxy', 'load-generator']
error_warning_logs = filtered_logs[
    (filtered_logs['content'].str.contains('error', case=False) | 
     filtered_logs['content'].str.contains('warning', case=False)) &
    (filtered_logs['_container_name_'].isin(services_of_interest))
]

error_warning_logs

```
Out[5]:
```
The `logs.parquet` file was successfully loaded, and logs were filtered to the alert time window (23:06:32 to 23:09:54 UTC+8). However, no error or warning messages related to the `frontend`, `frontend-proxy`, or `load-generator` services were found in the logs. This indicates that no obvious issues were logged for these services during the alert time window.

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [content, _time_, _source_, _namespace_, _pod_uid_, _container_ip_, _image_name_, _container_name_, _pod_name_, __topic__, __tag__:__pack_id__, __tag__:__hostname__, __tag__:_node_name_, __tag__:_node_ip_, __tag__:_cluster_id_]
Index: []

[0 rows x 15 columns]```
```